In [27]:
import osmnx as ox
from typing import List, Tuple, Set, Any
import pandas as pd
import geopandas as gpd
from itertools import combinations
from shapely.geometry import LineString
import xml.etree.ElementTree as ET
import os 
from dotenv import load_dotenv
import folium
import re
load_dotenv()

apikey = os.getenv("THUNDERFOREST_API_KEY")

In [28]:

def cluster_graph_by_nearest_grass(graph, grass_gdf, max_distance: int = 8) -> Tuple[gpd.GeoDataFrame, gpd.GeoDataFrame]:
    """
    Clusters graph nodes and edges based on the nearest grass polygon.

    Each node/edge within the max_distance of any grass polygon is assigned a
    'cluster_id' corresponding to the ID of the nearest grass patch.

    Args:
        nodes_gdf (GeoDataFrame): GeoDataFrame of graph nodes.
        edges_gdf (GeoDataFrame): GeoDataFrame of graph edges.
        grass_gdf (GeoDataFrame): GeoDataFrame of grass polygons.
        max_distance (int): The maximum distance in meters to consider an element
                            part of a cluster.

    Returns:
        tuple: A tuple containing two modified GeoDataFrames:
            - nodes_with_clusters (GeoDataFrame): Nodes with a new 'cluster_id' column.
            - edges_with_clusters (GeoDataFrame): Edges with a new 'cluster_id' column.
    """
    # --- Step 1: Preparation ---
    nodes_gdf, edges_gdf = ox.graph_to_gdfs(graph)
    
    # Create a clean, unique ID for each grass patch to use as the cluster ID.
    grass_with_id = grass_gdf.copy()
    grass_with_id['cluster_id'] = range(len(grass_with_id))

    # Project to a UTM CRS for accurate distance calculations
    utm_crs = nodes_gdf.estimate_utm_crs()
    nodes_proj = nodes_gdf.to_crs(utm_crs)
    edges_proj = edges_gdf.to_crs(utm_crs)
    grass_proj = grass_with_id.to_crs(utm_crs)

    # --- Step 2: Perform Nearest Neighbor Join ---
    
    # Use sjoin_nearest to find the single closest grass patch for each node and edge.
    # This is much more efficient than iterating.
    nodes_clustered_proj = gpd.sjoin_nearest(
        nodes_proj, grass_proj, how='left', max_distance=max_distance
    )
    edges_clustered_proj = gpd.sjoin_nearest(
        edges_proj, grass_proj, how='left', max_distance=max_distance
    )

    # --- Step 3: Map Cluster IDs back to original GeoDataFrames ---
    
    # Create the 'cluster_id' column in the original DataFrames.
    # It will be populated with NaN for elements not in any cluster.
    nodes_with_clusters = nodes_gdf.copy()
    edges_with_clusters = edges_gdf.copy()
    
    nodes_with_clusters['cluster_id'] = nodes_clustered_proj['cluster_id']
    edges_with_clusters['cluster_id'] = edges_clustered_proj['cluster_id']
    
    return nodes_with_clusters, edges_with_clusters

def generate_grass_paths(graph, grass_gdf, buildings_gdf) -> None:
    nodes_clustered, edges_clustered = cluster_graph_by_nearest_grass(graph, grass_gdf)
    
    original_nodes, original_edges = ox.graph_to_gdfs(graph)
    existing_edges_set: Set[frozenset] = set(
        frozenset(edge) for edge in original_edges.index.to_frame()[['u', 'v']].itertuples(index=False, name=None)
    )
    
    new_edges_list: List[dict] = []
    
    unique_cluster_ids = nodes_clustered['cluster_id'].dropna().unique()
    
    utm_crs = nodes_clustered.estimate_utm_crs()
    nodes_proj = nodes_clustered.to_crs(utm_crs)

    for cluster_id in unique_cluster_ids:
        cluster_nodes = nodes_clustered[nodes_clustered['cluster_id'] == cluster_id]
        
        # Only create edges if there's more than one node in the cluster.
        if len(cluster_nodes) > 1:
            # Create all unique pairs of nodes within the cluster.
            node_pairs = combinations(cluster_nodes.index, 2)
            
            for u, v in node_pairs:
                # Add edge only if it doesn't already exist in the original graph.
                if frozenset([u, v]) not in existing_edges_set:
                    geom_u = cluster_nodes.loc[u, 'geometry']
                    geom_v = cluster_nodes.loc[v, 'geometry']
                    
                    # Calculate accurate distance in meters.
                    dist_meters = nodes_proj.loc[u, 'geometry'].distance(nodes_proj.loc[v, 'geometry'])
                    
                    new_edges_list.append({
                        'u': u, 'v': v, 'key': 0,
                        'geometry': LineString([geom_u, geom_v]),
                        'surface': 'grass',
                        'length': round(dist_meters, 2),
                        'highway': 'path',
                        'generated': 1 # Flag to identify these edges later if needed
                    })
                    
    # Filter new edges that intersect with buildings
    if new_edges_list and not buildings_gdf.empty:
        # Create a temporary GeoDataFrame for efficient spatial operations.
        new_edges_gdf = gpd.GeoDataFrame(new_edges_list, crs=original_edges.crs)
        
        # Project both to a common UTM CRS.
        edges_proj = new_edges_gdf.to_crs(utm_crs)
        buildings_proj = buildings_gdf.to_crs(utm_crs)
        
        # Create a single unified geometry of all buildings for a huge performance gain.
        all_buildings_union = buildings_proj.unary_union
        
        # Create a boolean mask of edges that DO NOT intersect the building footprints.
        non_intersecting_mask = ~edges_proj.intersects(all_buildings_union)
        
        # Filter the GeoDataFrame to keep only the valid edges.
        valid_new_edges_gdf = new_edges_gdf[non_intersecting_mask]
        
    elif not new_edges_list:
        valid_new_edges_gdf = gpd.GeoDataFrame() # Ensure it's an empty GeoDataFrame
        
    else: # new_edges_list exists, but buildings_gdf is empty
        valid_new_edges_gdf = gpd.GeoDataFrame(new_edges_list, crs=original_edges.crs)

    # Add valid new edges to the original graph
    if not valid_new_edges_gdf.empty:
        # Set the index correctly for the new edges before concatenation.
        valid_new_edges_gdf = valid_new_edges_gdf.set_index(['u', 'v', 'key'])
        
        # Combine the original edges with the new, filtered grass path edges.
        final_edges_gdf = pd.concat([original_edges, valid_new_edges_gdf])
        
        print(f"Created new graph with {len(graph.edges())} total edges.")
        
        # Create the new, final graph from the original nodes and the combined edges.
        return ox.graph_from_gdfs(original_nodes, final_edges_gdf)
        
        

In [29]:
def remove_nodes_near_buildings(graph, buildings_gdf, buffer_distance=1.0):
    graph = graph.copy()
    if buildings_gdf.empty:
        print("No buildings found, skipping node removal")
        return
    
    # Get nodes from the graph
    nodes_gdf = ox.graph_to_gdfs(graph, edges=False)
    
    # ALWAYS convert to projected CRS for meter calculations
    target_crs = 'EPSG:3857'  # Web Mercator
    nodes_gdf_projected = nodes_gdf.to_crs(target_crs)
    buildings_gdf_projected = buildings_gdf.to_crs(target_crs)
    
    # Create buffer and find intersecting nodes
    buildings_buffered = buildings_gdf_projected.geometry.buffer(buffer_distance)
    buildings_union = buildings_buffered.unary_union
    
    nodes_to_remove = nodes_gdf_projected[
        nodes_gdf_projected.geometry.intersects(buildings_union)
    ]
    
    # Safety check
    removal_percentage = len(nodes_to_remove) / len(nodes_gdf) * 100
    print(f"Would remove {len(nodes_to_remove)} nodes ({removal_percentage:.1f}%)")
    
    if removal_percentage > 50:
        print("WARNING: Removing too many nodes! Check your buffer distance.")
        return
    
    # Remove nodes
    nodes_to_remove_ids = nodes_to_remove.index.tolist()
    
    print(f"Removed {len(nodes_to_remove_ids)} nodes from graph")
    graph.remove_nodes_from(nodes_to_remove_ids)
    
    return graph
        


In [19]:
def remove_problematic_relations(osm_file_path, output_path=None, missing_way_id="318135984"):
    """
    Remove relations that reference missing ways.
    """
    if output_path is None:
        output_path = osm_file_path.replace('.osm', '_fixed.osm')
    
    print(f"Removing problematic relations and saving to {output_path}...")
    
    try:
        tree = ET.parse(osm_file_path)
        root = tree.getroot()
        
        relations_removed = 0
        
        # Find and remove problematic relations
        for relation in root.findall('.//relation'):
            relation_id = relation.get('id')
            should_remove = False
            
            # Check if this relation references the missing way
            for member in relation.findall('member'):
                if member.get('type') == 'way' and member.get('ref') == missing_way_id:
                    should_remove = True
                    break
            
            if should_remove:
                root.remove(relation)
                relations_removed += 1
                print(f"Removed relation {relation_id}")
        
        # Write the cleaned XML
        tree.write(output_path, encoding='utf-8', xml_declaration=True)
        
        print(f"Removed {relations_removed} problematic relations")
        print(f"Clean OSM file saved as: {output_path}")
        
        return output_path
    
    except Exception as e:
        print(f"Error removing relations: {e}")
        return None



In [20]:
osm_file_path = "map.osm"

In [21]:
try:
    graph = ox.graph_from_xml(osm_file_path, simplify=False)
    
    edges_to_remove = []
    for u, v, k, d in graph.edges(keys=True, data=True):
        if not 'highway' in d or (d['highway'] not in ["footway", "path", "pedestrian", "residential", ...]):
            edges_to_remove.append((u, v, k))
    graph.remove_edges_from(edges_to_remove)
    
    isolated_nodes = [node for node, degree in dict(graph.degree()).items() if degree == 0]
    graph.remove_nodes_from(isolated_nodes)
    
    for i in range(10):
        try:
            gdf_all = ox.features.features_from_xml(osm_file_path)
            break
        except Exception as e:
            remove_problematic_relations(osm_file_path, osm_file_path, missing_way_id=str(e))
    else:
        print("Failed to fix osm file, there are still missing ways")
        raise
except Exception as e:
    print(f"Error loading graph from OSM file: {e}")
    raise

# Initialize empty GeoDataFrames for features
grass_gdf = gpd.GeoDataFrame()
buildings_gdf = gpd.GeoDataFrame()

# Try loading features with more relaxed error handling
try:
    # Load grass features
    grass_features = []
    for tags in [{'landuse': 'grass'}, 
                    {'landuse': 'recreation_ground'},
                    {'surface': 'grass'}, 
                    {'surface': 'lawn'},
                    {'surface': 'turf'},
                    {'natural': 'grassland'},
                    {'natural': 'grass'},
                    {'natural': 'lawn'},
                    {'natural': 'turf'},
                    {'leisure': 'garden'}]:
        try:
            gdf = ox.features_from_xml(osm_file_path, tags=tags)
            if not gdf.empty:
                grass_features.append(gdf)
        except Exception:
            continue
    
    if grass_features:
        grass_gdf = pd.concat(grass_features, ignore_index=True).drop_duplicates()
    else:
        print("No grass found")

    # Load buildings
    buildings = []

    for tags in [
                    {'building': 'yes'},
                    {'building': True},
                    {'building': 'house'},
                    {'building': 'residential'}, 
                    {'building': 'commercial'},
                    {'building': 'industrial'},
                    # Try without value specification
                    {'building': None}  # This sometimes works for any building tag
                ]:
        try:
            gdf = ox.features_from_xml(osm_file_path, tags=tags)
            if not gdf.empty:
                buildings.append(gdf)
        except Exception:
            continue
    
    if buildings:
        buildings_gdf = pd.concat(buildings, ignore_index=True).drop_duplicates()
    else:
        print("No buildings found")

except Exception as e:
    print(f"Warning: Could not load some features from OSM file: {e}")
    print("Proceeding with basic graph only")

# Remove nodes close to buildings
if not buildings_gdf.empty:
    graph = remove_nodes_near_buildings(graph, buildings_gdf)

if not grass_gdf.empty:
    graph = generate_grass_paths(graph, grass_gdf, buildings_gdf)
    
# remove duplicate edges
edges_to_remove: List[Any] = []
for edge in graph.edges:
    if edge[-1] > 0:
        edges_to_remove.append(edge)

graph.remove_edges_from(edges_to_remove)

Removing problematic relations and saving to map.osm...
Removed relation 18555106
Removed 1 problematic relations
Clean OSM file saved as: map.osm
Would remove 63 nodes (23.3%)
Removed 63 nodes from graph
Created new graph with 438 total edges.


In [32]:
minlat = None
minlon = None
maxlat = None
maxlon = None
with open(osm_file_path, 'r', encoding='utf-8') as f:
    lines = f.read().splitlines()
    for line in lines:
        if 'bounds' in line:
            temp = float(re.findall(r'[-+]?\d*\.\d+|\d+', line[line.find('minlat'):])[0])
            if minlat is None or minlat > temp:
                minlat = temp
            temp = float(re.findall(r'[-+]?\d*\.\d+|\d+', line[line.find('minlon'):])[0])
            if minlon is None or minlon > temp:
                minlon = temp
            temp = float(re.findall(r'[-+]?\d*\.\d+|\d+', line[line.find('maxlat'):])[0])
            if maxlat is None or maxlat < temp:
                maxlat = temp
            temp = float(re.findall(r'[-+]?\d*\.\d+|\d+', line[line.find('maxlon'):])[0])
            if maxlon is None or maxlon < temp:
                maxlon = temp

if minlat is None or minlon is None or maxlat is None or maxlon is None:
    print('Warning: impossible to get the map bounds from the OSM file,'
                     ' make sure the file contains the "bounds" tag.\n')
    raise ValueError('Impossible to get the map bounds from the OSM file')

# Define the projection
x = 0.5 * (maxlat + minlat)
y = 0.5 * (maxlon + minlon)

In [ ]:
tile_layers = {
    "OpenCycleMap": f"https://tile.thunderforest.com/cycle/{{z}}/{{x}}/{{y}}.png?apikey={apikey}",
    "Transport": f"https://tile.thunderforest.com/transport/{{z}}/{{x}}/{{y}}.png?apikey={apikey}",
    "Landscape": f"https://tile.thunderforest.com/landscape/{{z}}/{{x}}/{{y}}.png?apikey={apikey}",
    "Outdoors": f"https://tile.thunderforest.com/outdoors/{{z}}/{{x}}/{{y}}.png?apikey={apikey}",
    "Transport Dark": f"https://tile.thunderforest.com/transport-dark/{{z}}/{{x}}/{{y}}.png?apikey={apikey}",
    "Spinal Map": f"https://tile.thunderforest.com/spinal-map/{{z}}/{{x}}/{{y}}.png?apikey={apikey}",
    "Pioneer": f"https://tile.thunderforest.com/pioneer/{{z}}/{{x}}/{{y}}.png?apikey={apikey}",
    "Mobile Atlas": f"https://tile.thunderforest.com/mobile-atlas/{{z}}/{{x}}/{{y}}.png?apikey={apikey}",
    "Neighbourhood": f"https://tile.thunderforest.com/neighbourhood/{{z}}/{{x}}/{{y}}.png?apikey={apikey}",
    "Atlas": f"https://tile.thunderforest.com/atlas/{{z}}/{{x}}/{{y}}.png?apikey={apikey}",
}

# Create a folium map centered at your point
m = folium.Map(location=[x, y], zoom_start=15.5)

# Add your custom tile layers
for name, url in tile_layers.items():
    folium.TileLayer(
        tiles=url,
        attr='Thunderforest',
        name=name
    ).add_to(m)

# Add the grass areas to the map
folium.GeoJson(
    grass_gdf,
    style_function=lambda x: {
        'color': 'green',
        'weight': 2,
        'opacity': 0.8,
        'fillColor': 'lightgreen',
        'fillOpacity': 0.6
    }
).add_to(m)

# Add buildings to the map
folium.GeoJson(
    buildings_gdf,
    style_function=lambda x: {
        'color': 'yellow',
        'weight': 1,
        'opacity': 0.8,
        'fillColor': 'darkgray',
        'fillOpacity': 0.6
    }
).add_to(m)

# Convert graph to GeoDataFrames
nodes_gdf, edges_gdf = ox.graph_to_gdfs(graph)

# Add edges (streets) to the map
folium.GeoJson(
    edges_gdf,
    style_function=lambda x: {
        'color': 'blue',
        'weight': 2,
        'opacity': 0.8
    }
).add_to(m)

# Optionally add nodes (intersections) to the map
for idx, row in nodes_gdf.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3,
        color='red',
        fill=True,
        fillColor='red',
        fillOpacity=0.7,
        popup=f"Node: {idx}"
    ).add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

# Display the map
m

: 